In [ ]:
import pandas as pd

df = pd.read_csv("Resume.csv")
df = df[["ID", "Category", "Resume_str"]].rename(columns={"Resume_str": "text"})
df.head()

,ID,Category,text
0,16852973,HR,HR ADMINISTRATOR/MARKETING ASSOCIATE\...
1,22323967,HR,"HR SPECIALIST, US HR OPERATIONS ..."
2,33176873,HR,HR DIRECTOR Summary Over 2...
3,27018550,HR,HR SPECIALIST Summary Dedica...
4,17812897,HR,HR MANAGER Skill Highlights ...


In [ ]:
from bs4 import BeautifulSoup

jd_df = pd.read_csv("marketing_sample_for_trulia_com-real_estate__20190901_20191031__30k_data.csv")

jd_df = jd_df[["Uniq Id", "Job Title", "Job Description", "Industry", "City", "State"]]


def clean_html(text):
    if pd.isna(text):
        return ""
    return BeautifulSoup(text, "html.parser").get_text(separator=" ")

jd_df["clean_description"] = jd_df["Job Description"].apply(clean_html)

print(jd_df.shape)
print(jd_df[["Job Title", "clean_description"]].head(2))

(30002, 7)
                    Job Title  \
0               Shift Manager   
1  Operations Support Manager   

                                   clean_description  
0  WE ARE LOOKING FOR TOP PERFORMERS TO GROW WITH...  
1  \n \n JOB PURPOSE:  This position is responsib...  


In [ ]:
import re
from bs4 import BeautifulSoup

# --- Resume data ---
resume_df = pd.read_csv("Resume.csv")
resume_df = resume_df[["ID", "Category", "Resume_str"]].rename(columns={"Resume_str": "text"})

# --- JD data ---
jd_df = pd.read_csv("marketing_sample_for_trulia_com-real_estate__20190901_20191031__30k_data.csv")
jd_df = jd_df[["Uniq Id", "Job Title", "Job Description", "Industry", "City", "State"]]

def clean_html(text):
    if pd.isna(text):
        return ""
    return BeautifulSoup(text, "html.parser").get_text(separator=" ")

jd_df["text"] = jd_df["Job Description"].apply(clean_html)


def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    return text.strip()

resume_df["clean_text"] = resume_df["text"].apply(clean_text)
jd_df["clean_text"] = jd_df["text"].apply(clean_text)

print(resume_df.shape, jd_df.shape)

(2484, 4) (30002, 8)


In [ ]:
keywords = ["information technology", "software", "developer", "sales",
            "human resources", "hr ", "finance", "accountant", "teacher",
            "healthcare", "nurse", "engineer", "chef", "designer"]

pattern = "|".join(keywords)
jd_sample = jd_df[jd_df["Job Title"].str.lower().str.contains(pattern, na=False)].copy()
jd_sample = jd_sample.sample(n=min(300, len(jd_sample)), random_state=42).reset_index(drop=True)

print(jd_sample.shape)
jd_sample[["Job Title", "Industry"]].head(10)

(300, 8)


,Job Title,Industry
0,Software Programmer and Support - NAV,NaN
1,Sales Development Representative,NaN
2,Finance and Insurance Manager,NaN
3,Sales Associate/Beauty Advisor,NaN
4,Solution Design Consultants - Software (Multip...,NaN
5,Retail Sales Teammate,NaN
6,Additive Manufacturing Engineer,NaN
7,"Director of Sales, Courtyard Raleigh North/Tri...",NaN
8,IT Sales Consultant,NaN
9,Experienced Healthcare Insurance Verification ...,NaN


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


sample_resume = resume_df.iloc[0]
print("Resume Category:", sample_resume["Category"])


corpus = [sample_resume["clean_text"]] + jd_sample["clean_text"].tolist()

vectorizer = TfidfVectorizer(stop_words="english", max_features=5000)
tfidf_matrix = vectorizer.fit_transform(corpus)

resume_vector = tfidf_matrix[0]
jd_vectors = tfidf_matrix[1:]

similarities = cosine_similarity(resume_vector, jd_vectors).flatten()

jd_sample["match_score"] = similarities
top_matches = jd_sample.sort_values("match_score", ascending=False).head(5)
top_matches[["Job Title", "Industry", "match_score"]]

Resume Category: HR


,Job Title,Industry,match_score
45,Human Resources Generalist,NaN,0.218369
53,Sales & Marketing Coordinator,NaN,0.218291
62,General Sales Manager Automotive Tri State Area,NaN,0.161470
34,Director of Sales,NaN,0.158093
229,Recruiter / HR Generalist,NaN,0.151913


In [ ]:
def match_resume_to_jds(resume_text, jd_sample, top_n=5):
    corpus = [resume_text] + jd_sample["clean_text"].tolist()

    vectorizer = TfidfVectorizer(stop_words="english", max_features=5000)
    tfidf_matrix = vectorizer.fit_transform(corpus)

    resume_vector = tfidf_matrix[0]
    jd_vectors = tfidf_matrix[1:]

    similarities = cosine_similarity(resume_vector, jd_vectors).flatten()

    results = jd_sample.copy()
    results["match_score"] = similarities
    return results.sort_values("match_score", ascending=False).head(top_n)


def show_match(index):
    resume = resume_df.iloc[index]
    print(f"Resume Category: {resume['Category']}")
    print("-" * 50)
    matches = match_resume_to_jds(resume["clean_text"], jd_sample, top_n=5)
    return matches[["Job Title", "Industry", "match_score"]]

show_match(0)

Resume Category: HR
--------------------------------------------------


,Job Title,Industry,match_score
45,Human Resources Generalist,NaN,0.218369
53,Sales & Marketing Coordinator,NaN,0.218291
62,General Sales Manager Automotive Tri State Area,NaN,0.161470
34,Director of Sales,NaN,0.158093
229,Recruiter / HR Generalist,NaN,0.151913


In [ ]:

categories_to_test = ["INFORMATION-TECHNOLOGY", "HR", "SALES", "TEACHER", "FINANCE"]

for cat in categories_to_test:
    sample = resume_df[resume_df["Category"] == cat].iloc[0]
    print(f"\n=== Category: {cat} ===")
    matches = match_resume_to_jds(sample["clean_text"], jd_sample, top_n=3)
    print(matches[["Job Title", "match_score"]].to_string(index=False))


=== Category: INFORMATION-TECHNOLOGY ===
                                               Job Title  match_score
                                         Design Engineer     0.149152
                                     Engineering Manager     0.146049
DIRECTOR OF INFORMATION TECHNOLOGY SECURITY & COMPLIANCE     0.143707

=== Category: HR ===
                                      Job Title  match_score
                     Human Resources Generalist     0.218369
                  Sales & Marketing Coordinator     0.218291
General Sales Manager Automotive Tri State Area     0.161470

=== Category: SALES ===
                                    Job Title  match_score
Inside Sales/Technical Support Representative     0.172206
                              Sales Associate     0.167606
                           Solutions Engineer     0.162438

=== Category: TEACHER ===
                                                                      Job Title  match_score
      Educational Outside Sales

In [ ]:
!pip install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

jd_embeddings = model.encode(jd_sample["clean_text"].tolist(), show_progress_bar=True)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def match_resume_to_jds_embeddings(resume_text, jd_sample, jd_embeddings, top_n=5):
    resume_embedding = model.encode([resume_text])
    similarities = cosine_similarity(resume_embedding, jd_embeddings).flatten()

    results = jd_sample.copy()
    results["match_score"] = similarities
    return results.sort_values("match_score", ascending=False).head(top_n)

def show_match_embeddings(index):
    resume = resume_df.iloc[index]
    print(f"Resume Category: {resume['Category']}")
    print("-" * 50)
    matches = match_resume_to_jds_embeddings(resume["clean_text"], jd_sample, jd_embeddings, top_n=5)
    return matches[["Job Title", "match_score"]]

show_match_embeddings(0)

Resume Category: HR
--------------------------------------------------


,Job Title,match_score
45,Human Resources Generalist,0.701237
229,Recruiter / HR Generalist,0.641662
34,Director of Sales,0.606077
200,Sales Support Specialist,0.595291
103,Director of Sales and Marketing - Midwest loca...,0.565746


In [ ]:
for cat in categories_to_test:
    sample = resume_df[resume_df["Category"] == cat].iloc[0]
    print(f"\n=== Category: {cat} ===")
    matches = match_resume_to_jds_embeddings(sample["clean_text"], jd_sample, jd_embeddings, top_n=3)
    print(matches[["Job Title", "match_score"]].to_string(index=False))


=== Category: INFORMATION-TECHNOLOGY ===
                                               Job Title  match_score
DIRECTOR OF INFORMATION TECHNOLOGY SECURITY & COMPLIANCE     0.675125
                      Software Implementation Specialist     0.636944
                     Account Executive, Enterprise Sales     0.612733

=== Category: HR ===
                 Job Title  match_score
Human Resources Generalist     0.701237
 Recruiter / HR Generalist     0.641662
         Director of Sales     0.606077

=== Category: SALES ===
                                  Job Title  match_score
            Sales Representative, Metro Mix     0.642492
              Director of Sales & Marketing     0.607708
Relationship Manager - FI Sales (Mid-Level)     0.604839

=== Category: TEACHER ===
                                            Job Title  match_score
Macy's Coastland Center, Naples, FL: Sales Supervisor     0.447779
                             Sales Management Trainee     0.437265
               

In [ ]:
jd_sample[jd_sample["Job Title"].str.lower().str.contains("teach|instructor|tutor|professor")]

,Uniq Id,Job Title,Job Description,Industry,City,State,text,clean_text,match_score
212,a6a5e02cd73724f9ac8647961dc538a6,Plumbing Instructor / Trainer / Teacher,"<div id=""jobDescriptionText"" class=""jobsearch-...",NaN,Orange,CA,"\n \n Do It Right Plumbers of Orange, CA is...",do it right plumbers of orange ca is looking ...,0.088049


In [ ]:
keywords = ["information technology", "software", "developer", "sales",
            "human resources", "hr generalist", "finance", "accountant",
            "teacher", "teaching", "instructor", "tutor", "professor", "educator",
            "healthcare", "nurse", "engineer", "chef", "designer"]

pattern = "|".join(keywords)
jd_sample = jd_df[jd_df["Job Title"].str.lower().str.contains(pattern, na=False)].copy()

teaching_check = jd_sample[jd_sample["Job Title"].str.lower().str.contains("teach|instructor|tutor|professor|educator")]
print(f"Teaching jobs found: {len(teaching_check)}")

jd_sample = jd_sample.sample(n=min(500, len(jd_sample)), random_state=42).reset_index(drop=True)
print(jd_sample.shape)

Teaching jobs found: 57
(500, 8)


In [ ]:
jd_embeddings = model.encode(jd_sample["clean_text"].tolist(), show_progress_bar=True)

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

In [ ]:
for cat in categories_to_test:
    sample = resume_df[resume_df["Category"] == cat].iloc[0]
    print(f"\n=== Category: {cat} ===")
    matches = match_resume_to_jds_embeddings(sample["clean_text"], jd_sample, jd_embeddings, top_n=3)
    print(matches[["Job Title", "match_score"]].to_string(index=False))


=== Category: INFORMATION-TECHNOLOGY ===
                           Job Title  match_score
Information Technology Administrator     0.626863
 Director Information Technology-AAM     0.604627
                   Software Engineer     0.599544

=== Category: HR ===
                    Job Title  match_score
             Sales Supervisor     0.624757
Director of Sales & Marketing     0.607864
   Assistant Finance Director     0.606299

=== Category: SALES ===
                                Job Title  match_score
Account Manager - Direct Sales - SLG/K-12     0.680783
                   Sales Assistant - WTVF     0.634274
                    Sales Account Manager     0.632832

=== Category: TEACHER ===
                       Job Title  match_score
         Sr Technical Instructor     0.581030
         Sr Technical Instructor     0.581030
Sales & Customer Success Manager     0.515141

=== Category: FINANCE ===
                 Job Title  match_score
          Staff Accountant     0.645140
A

In [ ]:
print("Duplicate rows in jd_sample:", jd_sample.duplicated(subset=["Job Title", "clean_text"]).sum())
jd_sample = jd_sample.drop_duplicates(subset=["Job Title", "clean_text"]).reset_index(drop=True)
print("New shape:", jd_sample.shape)

Duplicate rows in jd_sample: 38
New shape: (462, 8)


In [ ]:
hr_jobs = jd_sample[jd_sample["Job Title"].str.lower().str.contains("human resources|hr generalist|hr manager|recruiter")]
print(f"HR jobs in sample: {len(hr_jobs)}")
print(hr_jobs[["Job Title"]])

HR jobs in sample: 5
                             Job Title
100                Field HR Generalist
155          Corporate Sales Recruiter
363           Software Sales Recruiter
432  Senior Human Resources Consultant
445             Human Resources Expert


In [ ]:
category_keywords = {
    "IT": ["software", "information technology", "developer", "programmer"],
    "HR": ["human resources", "hr generalist", "hr manager", "recruiter", "talent acquisition"],
    "SALES": ["sales representative", "sales manager", "account executive"],
    "FINANCE": ["finance", "accountant", "financial analyst"],
    "TEACHER": ["teacher", "instructor", "tutor", "professor", "educator"],
}

samples = []
for cat, kws in category_keywords.items():
    pattern = "|".join(kws)
    matched = jd_df[jd_df["Job Title"].str.lower().str.contains(pattern, na=False)]

    matched = matched.drop_duplicates(subset=["Job Title"])

    if not matched.empty:
        matched = matched.sample(n=min(50, len(matched)), random_state=42)
    else:

        matched = pd.DataFrame(columns=matched.columns)

    samples.append(matched)
    print(f"{cat}: {len(matched)} jobs found")

jd_sample = pd.concat(samples).reset_index(drop=True)
print("\nFinal jd_sample shape:", jd_sample.shape)

IT: 50 jobs found
HR: 50 jobs found
SALES: 50 jobs found
FINANCE: 50 jobs found
TEACHER: 29 jobs found

Final jd_sample shape: (229, 8)


In [ ]:
jd_embeddings = model.encode(jd_sample["clean_text"].tolist(), show_progress_bar=True)

for cat in categories_to_test:
    sample = resume_df[resume_df["Category"] == cat].iloc[0]
    print(f"\n=== Category: {cat} ===")
    matches = match_resume_to_jds_embeddings(sample["clean_text"], jd_sample, jd_embeddings, top_n=3)
    print(matches[["Job Title", "match_score"]].to_string(index=False))

Batches:   0%|          | 0/8 [00:00<?, ?it/s]


=== Category: INFORMATION-TECHNOLOGY ===
                                                                 Job Title  match_score
                                                   VP Software Development     0.681104
                                      Information Technology Administrator     0.626863
Cyber Identity & Access Management Senior Consultant - SailPoint Developer     0.613545

=== Category: HR ===
                                              Job Title  match_score
                             Human Resources Generalist     0.664722
Human Resources Manager- Greensboro, NC Friendly Center     0.653776
                              Recruiter / HR Generalist     0.641662

=== Category: SALES ===
                                           Job Title  match_score
     Territory Sales Manager (Northern South Dakota)     0.649280
         Inside Account Executive | Higher Education     0.642703
Macy's Rosedale Center, Roseville, MN: Sales Manager     0.630985

=== Category: TEAC

In [ ]:
SKILLS_LIST = [
    # Tech
    "python", "java", "javascript", "sql", "excel", "aws", "azure", "docker",
    "kubernetes", "machine learning", "data analysis", "power bi", "tableau",
    "html", "css", "react", "node.js", "git", "linux", "agile", "scrum",
    # Business/Finance
    "accounting", "budgeting", "forecasting", "financial analysis", "quickbooks",
    "sap", "erp", "audit", "taxation", "bookkeeping",
    # HR
    "recruiting", "onboarding", "payroll", "employee relations", "talent acquisition",
    "performance management", "hris", "labor relations",
    # Sales/Marketing
    "crm", "salesforce", "negotiation", "lead generation", "cold calling",
    "digital marketing", "seo", "social media", "customer service",
    # Teaching
    "curriculum development", "lesson planning", "classroom management",
    "special education", "student assessment",
    # Soft skills
    "leadership", "communication", "project management", "team management",
    "problem solving", "time management", "training", "presentation"
]

print(f"Total skills tracked: {len(SKILLS_LIST)}")

Total skills tracked: 61


In [ ]:
def extract_skills(text, skills_list=SKILLS_LIST):
    text_lower = text.lower()
    found_skills = [skill for skill in skills_list if skill in text_lower]
    return set(found_skills)

In [ ]:
def skill_gap_analysis(resume_text, jd_text, skills_list=SKILLS_LIST):
    resume_skills = extract_skills(resume_text, skills_list)
    jd_skills = extract_skills(jd_text, skills_list)

    matched_skills = resume_skills & jd_skills
    missing_skills = jd_skills - resume_skills
    extra_skills = resume_skills - jd_skills

    return {
        "matched_skills": sorted(matched_skills),
        "missing_skills": sorted(missing_skills),
        "extra_skills": sorted(extra_skills)
    }

In [ ]:
def full_match_report(resume_index, jd_sample, jd_embeddings, top_n=1):
    resume = resume_df.iloc[resume_index]
    print(f"Resume Category: {resume['Category']}")
    print("=" * 60)

    # Top match
    top_match = match_resume_to_jds_embeddings(
        resume["clean_text"], jd_sample, jd_embeddings, top_n=top_n
    ).iloc[0]

    print(f"Best Matching Job: {top_match['Job Title']}")
    print(f"Match Score: {top_match['match_score']:.3f}")
    print("-" * 60)

    # Skill gap
    gaps = skill_gap_analysis(resume["text"], top_match["text"])

    print(f"✅ Matched Skills ({len(gaps['matched_skills'])}):")
    print(", ".join(gaps['matched_skills']) if gaps['matched_skills'] else "None found")

    print(f"\n❌ Missing Skills - JD requires but resume lacks ({len(gaps['missing_skills'])}):")
    print(", ".join(gaps['missing_skills']) if gaps['missing_skills'] else "None missing!")

    print(f"\n➕ Extra Skills - resume has but JD doesn't mention ({len(gaps['extra_skills'])}):")
    print(", ".join(gaps['extra_skills']) if gaps['extra_skills'] else "None")

# Test
full_match_report(resume_df[resume_df["Category"] == "INFORMATION-TECHNOLOGY"].index[0], jd_sample, jd_embeddings)

Resume Category: INFORMATION-TECHNOLOGY
Best Matching Job: VP Software Development
Match Score: 0.681
------------------------------------------------------------
✅ Matched Skills (5):
communication, erp, excel, linux, training

❌ Missing Skills - JD requires but resume lacks (5):
accounting, java, leadership, sql, talent acquisition

➕ Extra Skills - resume has but JD doesn't mention (1):
html
